In [ ]:
import os
os.environ["POLARS_FORCE_NEW_STREAMING"]="1"
import datetime
import colormaps
from pathlib import Path
from functools import partial
from itertools import product
from string import ascii_lowercase
from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.colors import (
    LinearSegmentedColormap,
    BoundaryNorm,
    Normalize,
    rgb_to_hsv,
    hsv_to_rgb,
)
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs
import PIL

os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"
os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import (
    PRETTIER_VARNAME,
    YEARS,
    UNITS,
    RESULTS,
    FACTORS_UNITS,
    FACTORS,
    DATADIR,
    DUNCANS_REGIONS_NAMES,
    MONTH_NAMES,
    FIGURES,
    RADIUS,
    get_region,
    squarify,
    polars_to_xarray,
    xarray_to_polars,
    to_expr,
    get_index_columns,
)
from jetutils.data import standardize, standardize_polars_dtypes, compute_anomalies_pl
from jetutils.geospatial import (
    central_diff,
    haversine_from_dl,
    compute_relative_clim,
    compute_relative_sm,
    compute_relative_std,
    compute_relative_anom,
    create_all_relative_plots,
)
from jetutils.jet_finding import (
    average_jet_categories,
    track_jets,
    slowness_from_cross,
    spells_from_cross,
    connected_from_cross,
    slowness_expr,
    spells_from_cross_catd_simple,
    extract_features,
    gaussian_smooth_func,
    find_all_jets,
    jet_position_as_da,
    to_one_large,
)
from jetutils.plots import (
    STYLE_SHEET,
    COLORS,
    COLORS_EXT,
    WERNLI_FLAIR,
    WERNLI_FLAIR_LEVELS,
    Clusterplot,
    plot_interp,
    plot_relative_time,
    gaussian_kde,
    props_vs_quantiles,
)
from jetutils.anyspell import (
    make_daily,
    mask_from_spells_pl,
    subset_around_onset,
    extend_spells,
    get_spells,
)
from jetutils.stats import (
    create_bootstrapped_times,
    odds_ratio,
    is_signif_OR,
    common_OR,
)

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}
# pl.Config.set_verbose(True)
# pl.Config.set_streaming_chunk_size(500)  
basepath = Path(f"{DATADIR}/exp9")

ALL_TIMES = (
    pl.datetime_range(
        start=pl.datetime(1959, 1, 1),
        end=pl.datetime(2023, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9]))
summer = summer_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)

In [ ]:
jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
phat_jets = to_one_large(jets)
props = pl.read_parquet(basepath.joinpath("props.parquet"))
cross = pl.read_parquet(basepath.joinpath("cross.parquet"))
pers = slowness_from_cross(cross).drop("is_polar")
props = props.join(pers, on=["time", "jet ID"])

over_europe = pl.col("lon") > -10
lat_over_europe = (pl.col("lat") * pl.col("s")).filter(over_europe).sum() / pl.col(
    "s"
).filter(over_europe).sum()
lat_over_europe = jets.group_by("time", "jet ID").agg(
    lat_over_europe.fill_nan(0).alias("lat_over_europe")
)
props = props.join(lat_over_europe, on=["time", "jet ID"])

props_catd = squarify(average_jet_categories(props), ["time", "jet"])
props_catd = props_catd.join(
    props_catd.rolling("time", period="2d", group_by="jet").agg(
        **{
            f"{col}_var": pl.col(col).var()
            for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
        }
    ),
    on=["time", "jet"],
)
pers = props_catd.rolling("time", period="5d", group_by="jet").agg(
    pers=pl.col("slowness").sum()
)
props_catd = props_catd.join(pers, on=("time", "jet"))
props_catd_summer = summer_filter.join(props_catd, on="time")

dist_threshold = 2.0e6
overlap_threshold = 0.6
dis_polar_thresh = 0.12
spells_list = spells_from_cross(
    jets,
    cross,
    None,
    dist_threshold,
    overlap_threshold,
    dis_polar_thresh,
    n_STJ=30,
    n_EDJ=30,
    season=summer,
    STJ_lat_threshold=30,
)
_, summary_comp = connected_from_cross(
    jets,
    cross,
    dist_threshold,
    overlap_threshold,
    dis_polar_thresh,
)
jet_column = (
    pl.when((pl.col("is_polar") > 0.5).mean().over("spell") > 0.5)
    .then(pl.lit("EDJ"))
    .otherwise(pl.lit("STJ"))
)
summary_comp = (
    summary_comp.filter(pl.col("len") > 1)
    .drop("s", "theta", "is_polar", "s_right", "theta_right", "is_polar_right")
    .join(props["time", "jet ID", "is_polar"], on=("time", "jet ID"))
    .with_columns(
        jet=jet_column,
        slowness=slowness_expr(),
    )
    .with_columns(
        slowness_sum=pl.col("slowness").sum().over("spell"),
    )
    # .join(props, on=["time", "jet ID"], suffix="_fromprops")
    .sort("time", "jet ID")
)
summer_summary_comp = summary_comp.filter(
    pl.col("time")
    .is_in(pl.lit(summer.implode().first(), pl.List(pl.Datetime("ms"))))
    .over("spell")
)

for jet_name, spells in spells_list.items():
    print(jet_name, spells["spell"].n_unique())
    print(jet_name, spells["len"].min() / 4)

daily_spells_list = {
    a: make_daily(b, "spell", ["len", "spell_of"]) for a, b in spells_list.items()
}

In [ ]:
from jetutils.geospatial import interp_jets_to_zero_one, directional_diff
jets_with_interp = pl.read_parquet(basepath.joinpath("playtest.parquet")).drop("angle")
# jets_with_interp = jets_with_interp.with_columns(pl.col("n") / 1e6, pl.col("s_interp") / 10)
filter_centre = pl.col("n").abs() <= 8e5
n_f = pl.col("n").filter(filter_centre)
s_f = pl.col("s_interp").filter(filter_centre)
u_f = pl.col("u_interp").filter(filter_centre)
v_f = pl.col("v_interp").filter(filter_centre)
# mu = (n_f * s_f).sum() / s_f.sum()
mu = n_f.get(s_f.arg_max())
sigmasq = ((n_f - mu) ** 2 * s_f).sum() / s_f.sum()
# sigmasq = ((n_f - mu) ** 2).mean()
factor = pl.col("s_interp").filter(filter_centre).max()
angle = pl.arctan2(v_f.get(s_f.arg_max()), u_f.get(s_f.arg_max()))
gparams = jets_with_interp.group_by("time", "jet ID", "index", maintain_order=True).agg(
    mu=mu, sigmasq=sigmasq, factor=factor, angle=angle
)
# gparams = gparams.rolling("index", group_by=["time", "jet ID"], period="11i", offset="-6i").agg(pl.mean("mu", "sigmasq", "factor"), pl.col("angle").get(-6))
jets_with_interp = jets_with_interp.join(gparams, on=["time", "jet ID", "index"])
gs_expr = pl.col("factor") * (-(pl.col("n") - pl.col("mu")).pow(2) / pl.col("sigmasq") * 0.5).exp()
# gs_expr = pl.min_horizontal(gs_expr, pl.col("s_interp"))
u_gaussian = gs_expr * pl.col("angle").cos()
v_gaussian = gs_expr * pl.col("angle").sin()

jets_with_interp = jets_with_interp.with_columns(s_gaussian=gs_expr, u_gaussian=u_gaussian, v_gaussian=v_gaussian)
fake_eke = (pl.col("up") ** 2 + pl.col("vp") ** 2) * 0.5
fake_emf = pl.col("up") * pl.col("vp")
fake_emfc = -central_diff(fake_emf) / central_diff("n")
fake_eke = jets_with_interp.select("time", "jet ID", "index", "n", "is_polar", "normallon_rounded", "normallat_rounded", up=pl.col("u_interp") - pl.col("u_gaussian"), vp=pl.col("v_interp") - pl.col("v_gaussian")).with_columns(fake_eke=fake_eke, fake_emf=fake_emf, fake_emfc=fake_emfc)
fake_eke = interp_jets_to_zero_one(fake_eke, ["normallon_rounded", "normallat_rounded", "up", "vp", "fake_eke", "fake_emf", "fake_emfc"])
# jets_with_interp = jets_with_interp.join(jets_with_interp.filter(pl.col("n").abs() <= 3e5).select("time", "jet ID", "index", "n", "s_interp", "s_centre"), on=["time", "jet ID", "index"])

In [ ]:
da_df = polars_to_xarray(fake_eke, ["time", "jet ID", "n", "norm_index"])

In [ ]:
da_df.mean(["time", "jet ID"])["fake_emfc"].plot()

In [ ]:
polars_to_xarray(pl.read_parquet(basepath.joinpath("EKE250_5days_relative.parquet")).filter(pl.col("time").dt.year() == 1959), ["time", "jet ID", "n", "norm_index"])["EKE250_5days_interp"].mean(["time", "jet ID"]).plot()

In [ ]:
n_t, n_i = 460, 19
da_df["s_interp"][n_t, 1, :, n_i].plot()
da_df["s_gaussian"][n_t, 1, :, n_i].plot()

In [ ]:
# https://math.stackexchange.com/questions/441422/gaussian-curve-fitting-parameter-estimation
# ndiff = pl.col("n").diff()
# S_expr = 0.5 * (pl.col("s_interp") + pl.col("s_interp").shift(1)) * ndiff
# S_expr = S_expr.fill_null(0).cum_sum()
# T_expr = 0.5 * (pl.col("s_interp") * pl.col("n") + pl.col("s_interp").shift(1) * pl.col("n").shift(1)) * ndiff
# T_expr = T_expr.fill_null(0).cum_sum()
# m_11 = S_expr.pow(2).sum()
# m_22 = T_expr.pow(2).sum()
# m_offd = (T_expr * S_expr).sum()
# mat_det = 1 / (m_11 * m_22 - m_offd.pow(2)) 
# mat_inv_11 = mat_det * m_22
# mat_inv_22 = mat_det * m_11
# mat_inv_offd = - mat_det * m_offd
# b1 = (S_expr * (pl.col("s_interp") - pl.col("s_interp").first())).sum()
# b2 = (T_expr * (pl.col("s_interp") - pl.col("s_interp").first())).sum()
# A = mat_inv_11 * b1 + mat_inv_offd * b2
# B = mat_inv_offd * b1 + mat_inv_22 * b2
# a = - A / B
# b = - 2 / B
# c = pl.col("s_interp").sum() / (-(pl.col("n") - a).pow(2) / b).exp().sum()
# # dummy_exp = pl.DataFrame({
# #     "n": np.arange(1, 15),
# #     "s_interp": [0.84, 1.28, 1.72, 1.82, 1.93, 2.01, 1.64, 1.19, 0.92, 0.6, 0.29, 0.17, 0.09, 0.01],
# # })
# # dummy_exp.with_columns(a=a, b=b, c=c)


In [ ]:
period = 11
offset = int(np.ceil(period / 2))
bees = (
    jets_with_interp
    .select("time", "jet ID", "index", "n", s_ratio=pl.col("s_interp") / pl.col("s_centre"))
    .filter(pl.col("n") != 0, pl.col("n").abs() < 7e5)
    .filter(pl.col("s_ratio") < 1)
    .with_columns(side=pl.col("n").sign(), b=pl.col("n").abs() / pl.col("s_ratio").log().abs().sqrt())
    .group_by("time", "jet ID", "index", "side", maintain_order=True)
    .agg(pl.col("b").mean())
    # .rolling("index", group_by=["time", "jet ID", "side"], period=f"{period}i", offset=f"-{offset}i")
    # .agg(pl.col("b").mean())
    .sort("time", "jet ID", "index", "side")
)
bees = (
    squarify(bees, [("time", "jet ID", "index"), "side"])
    .with_columns(pl.col("b").fill_nan(None).fill_null(strategy="mean").over(["time", "jet ID", "index"]))
)
gs_expr = pl.col("s_centre") * (-pl.col("n").pow(2) / pl.col("b").pow(2)).exp()
jets_with_interp = (
    jets_with_interp
    .with_columns(side=pl.col("n").sign().cast(pl.Float32()))
    .join(bees, on=["time", "jet ID", "index", "side"])
    .with_columns(s_gaussian=gs_expr)
)

In [ ]:
smoothed = []
for nabs, jwi in jets_with_interp.group_by(pl.col("n").abs()):
    nabs = nabs[0]
    period = int(nabs / 1e5)
    offset = int(np.ceil(period / 2))
    if nabs == 0:
        smoothed.append(jwi.select("time", "jet ID", "n", "index", "s_interp"))
        continue
    break
    jwi = (
        jwi
        .rolling("index", group_by=("time", "jet ID", "n"), period=f"{period}i", offset=f"-{offset}i")
        .agg(pl.col("s_interp").mean())
    )
    smoothed.append(jwi)
# smoothed = pl.concat(smoothed).sort("time", "jet ID", "index", "n")
# jets_with_interp = jets_with_interp.join(smoothed.rename({"s_interp": "s_smooth"}), on=["time", "jet ID", "index", "n"])

In [ ]:
jwi